# Desarrollo de un sistema RAG para recomendación basada en contenido

El conjunto de datos elegido es **[Anime Dataset 2023](https://www.kaggle.com/datasets/dbdmobile/myanimelist-dataset)**, obtenido de la plataforma Kaggle. 

Este *dataset* contiene información exhaustiva sobre un gran catálogo de animes, lo cual resulta ideal para desarrollar **sistemas de recomendación basados en contenido**. Asimismo, incluye datos sobre diferentes usuarios y las calificaciones (*ratings*) que han dado a diversos animes, lo que permitirá construir perfiles de usuario para evaluar y poner a prueba el sistema de recomendación.

En concreto, el archivo principal, `anime-dataset-2023.csv`, está compuesto por las siguientes columnas:

* **`anime_id`**: Identificador único para cada anime.
* **`Name`**: Nombre del anime en su idioma original.
* **`English name`**: Nombre oficial del anime en inglés.
* **`Other name`**: Nombre o título nativo del anime (puede estar en japonés, chino o coreano).
* **`Score`**: Puntuación o calificación media del anime.
* **`Genres`**: Géneros a los que pertenece el anime, separados por comas.
* **`Synopsis`**: Breve descripción o resumen de la trama del anime.
* **`Type`**: Formato o tipo de emisión del anime (ej. serie de TV, película, OVA, ONA, etc.).
* **`Episodes`**: Número total de episodios que componen el anime.
* **`Aired`**: Fechas de inicio y fin de la emisión del anime.
* **`Premiered`**: Temporada y año en que se estrenó el anime (ej. *spring 2023*).
* **`Status`**: Estado actual de emisión del anime (ej. Finalizado, En emisión, No emitido aún).
* **`Producers`**: Empresas de producción.
* **`Licensors`**: Empresas que poseen las licencias de distribución del anime (ej. plataformas de *streaming*).
* **`Studios`**: Estudios de animación encargados de producir el anime.
* **`Source`**: Material original en el que se basa la obra (ej. manga, novela ligera, original).
* **`Duration`**: Duración aproximada de cada episodio.
* **`Rating`**: Clasificación por edades recomendada para el anime.
* **`Rank`**: Posición (*ranking*) general del anime basada en su puntuación.
* **`Popularity`**: Posición del anime en el ranking de popularidad general.
* **`Favorites`**: Número de veces que el anime ha sido marcado como "favorito" por los usuarios.
* **`Scored By`**: Cantidad total de usuarios que han calificado el anime.
* **`Members`**: Número de usuarios que han añadido el anime a sus listas personales en la plataforma.
* **`Image URL`**: Enlace a la imagen o póster promocional del anime.

Por otro lado, el archivo `users-score-2023.csv` contiene la siguiente información:
* **user_id**: ID único para cada usuario.
* **Username**: el nombre de usuario del usuario.
* **anime_id**: ID único para cada anime.
* **Anime Title**: título del anime.
* **rating**: la puntuació dada al anime por el usuario.

## Sesión 1

Procesado de ítems, generación de embeddings e indexación.

In [1]:
import pandas as pd
from sentence_transformers import SentenceTransformer
import numpy as np
import faiss

c:\Users\Laura\anaconda3\envs\rag\Lib\site-packages\tqdm\auto.py:21: TqdmWarning: IProgress not found. Please update jupyter and ipywidgets. See https://ipywidgets.readthedocs.io/en/stable/user_install.html
  from .autonotebook import tqdm as notebook_tqdm


### Cargar los datos

En primer lugar, cargamos el conjunto de datos.

In [2]:
# Cargamos el dataset
df = pd.read_csv('data/anime-dataset-2023.csv')

### Preprocesamiento

El conjunto de datos tiene valores faltantes, pero no figuran como tal ya que están representados con cadenas del tipo "UNKNOWN" o "Not available", así que los remplazaremos con valores NaN reales para poder gestionarlos más fácilmente.

In [3]:
# Strings que representan valor nulo
NULL_STRINGS = [
    'Unknown',
    '[null]',
    'UNKNOWN',
    'unknown',
    'Not available',
    '',
    'No description available for this anime.'
]

# Reemplazar todos esos valores por NaN real
df.replace(NULL_STRINGS, np.nan, inplace=True)

print('Tamaño del dataset:', df.shape)
print('\nValores nulos por columna:') 
print(df.isnull().sum())

Tamaño del dataset: (24905, 24)

Valores nulos por columna:
anime_id            0
Name                0
English name    14577
Other name        128
Score            9213
Genres           4929
Synopsis         4535
Type               74
Episodes          611
Aired             915
Premiered       19399
Status              0
Producers       13350
Licensors       20170
Studios         10526
Source           3689
Duration          663
Rating            669
Rank             4612
Popularity          0
Favorites           0
Scored By        9213
Members             0
Image URL           0
dtype: int64


Vemos que hay multitud de valores faltantes, siendo especialmente críticos los valores faltantes en la columna `Synopsis`, ya que es la principal fuente de contenido textual y semántico. Por ello y porque el total de filas con valor nulo en este campo no representa un porcentaje demasiado elevado con respecto al total, se optará por eliminar las filas que no tengan valor para esta columna.

In [4]:
df.dropna(subset=['Synopsis'], inplace=True)
print('Tamaño del dataset después de eliminar filas sin sinopsis:', df.shape)

Tamaño del dataset después de eliminar filas sin sinopsis: (20370, 24)


Tras eliminar los registros sin sinopsis, el conjunto de datos sigue contando con más de 20.000 animes, lo cual sigue siendo un volumen de datos suficientemente amplio y representativo para desarrollar un sistema de recomendación.

Para gestionar los valores nulos restantes sin reducir más el tamaño del *dataset*, aplicaremos una imputación simple. El título en inglés se rellenará utilizando el nombre original del anime (`Name`), mientras que al resto de variables categóricas clave se les asignará una etiqueta genérica indicando que el dato es desconocido.

In [5]:
# English name: si falta, usar el nombre original
df['English name'] = df['English name'].fillna(df['Name'])

# Resto de columnas
df['Genres'] = df['Genres'].fillna('genre unknown')
df['Type'] = df['Type'].fillna('type unknown')
df['Studios'] = df['Studios'].fillna('studio unknown')
df['Source'] = df['Source'].fillna('source unknown')
df['Rating'] = df['Rating'].fillna('rating unknown')

A continuación, transformaremos la variable `Episodes` en una variable categórica. Agrupar la cantidad exacta de episodios en rangos discretos facilitará al modelo la tarea de encontrar similitudes basadas en el formato y la duración de las obras.

Dado que utilizaremos modelos de lenguaje para generar *embeddings* de texto, los números exactos aislados pierden su valor matemático y aportan poco contexto. Convertir la cantidad de episodios a categorías textuales (como *short series* o *long series*) permite que el modelo interprete semánticamente la duración de la obra.

In [6]:
def categorize_episodes(ep):
    try:
        ep = int(ep)
        if ep == 1:
            return 'single episode / movie / special'
        elif ep <= 15:
            return 'short series'
        elif ep <= 50:
            return 'medium series'
        elif ep <= 100:
            return 'long series'
        else:
            return 'very long series'
    except:
        return 'unknown length'

df['Episodes'] = df['Episodes'].apply(categorize_episodes)

Siguiendo la misma lógica aplicada a los episodios, extraeremos el año de emisión para agrupar los animes por décadas.

In [7]:
# Extraer año
df['Year'] = df['Aired'].str.extract(r'(\d{4})').astype('Int64') # extrae el primer año que encuentra, si no hay, queda como NaN

# Definir rangos
bins = [0, 1989, 1999, 2009, 2019, 2029]
labels = [
    'classic anime',
    '90s anime',
    '2000s anime',
    '2010s anime',
    '2020s anime'
]

# Categorizar
df['Era'] = pd.cut(df['Year'], bins=bins, labels=labels)

# Forzar "unknown era" en valores faltantes
df['Era'] = df['Era'].cat.add_categories(['unknown era']).fillna('unknown era')

A continuación, normalizaremos todas las columnas textuales y categóricas. Eliminaremos los espacios en blanco adicionales y convertiremos todo el texto a minúsculas.

In [8]:
# Limpiar espacios y pasar a minúsculas en las columnas de texto
TEXT_COLS = ['Name', 'English name', 'Genres', 'Synopsis', 'Type', 'Source', 'Studios', 'Rating']

for col in TEXT_COLS:
    df[col] = df[col].apply(lambda x: ' '.join(str(x).split()).lower())

Finalmente, procederemos a construir el "documento" unificado para cada anime, concatenando la sinopsis y los metadatos descriptivos en una única cadena de texto estructurada. Las columnas utilizadas son las siguientes:
- `English name`
- `Genres`
- `Type`
- `Episodes`
- `Era`
- `Source`
- `Studios`
- `Rating`
- `Synopsis`

Se han seleccionado exclusivamente estas variables porque capturan la identidad, temática, formato y contexto de producción de la obra. Por otro lado, se han descartado las métricas numéricas y de interacción comunitaria (como popularidad, favoritos o puntuaciones) para garantizar un enfoque puramente basado en contenido, evitando así introducir sesgos o ruido. Este texto resultante será la entrada que utilizará el modelo de lenguaje para generar los *embeddings*.

In [9]:
def build_document(row):
    return (
        f"title: {row['English name']}. "
        f"genres: {row['Genres']}. "
        f"type: {row['Type']}. "
        f"episodes: {row['Episodes']}. "
        f"era: {row['Era']}. "
        f"source: {row['Source']}. "
        f"studio: {row['Studios']}. "
        f"rating: {row['Rating']}. "
        f"synopsis: {row['Synopsis']}."
    )
df['document'] = df.apply(build_document, axis=1)

Para garantizar la compatibilidad con el modelo de *embeddings* seleccionado (`all-MiniLM-L6-v2`), analizaremos la distribución de la longitud de nuestros documentos. Dado que el modelo tiene una ventana de contexto de 256 *tokens*, este cálculo estadístico nos permitirá comprobar qué porcentaje de nuestro catálogo encaja en este límite y confirmar si es necesario aplicar técnicas de *chunking* (troceado) o si el truncamiento automático del modelo es suficiente.

In [10]:
df['doc_length'] = df['document'].apply(lambda x: len(x.split()))
print(df['doc_length'].describe())

count    20370.000000
mean        93.235788
std         58.969977
min         25.000000
25%         44.000000
50%         75.000000
75%        129.000000
max        666.000000
Name: doc_length, dtype: float64


In [11]:
# Calcular el percentil 98.5 usando la columna doc_length
percentil_98_5 = df['doc_length'].quantile(0.985)

print(f"El percentil 98.5 de la longitud de los documentos es: {percentil_98_5:.0f} palabras")

El percentil 98.5 de la longitud de los documentos es: 243 palabras


Como se puede observar en las estadísticas, la longitud media de los documentos es de apenas 99 palabras y el 75% del catálogo no supera las 135. Además, el cálculo del percentil 98.5 confirma que casi la totalidad del *dataset* se encuentra por debajo de las 243 palabras. Dado que el modelo generador de *embeddings* `all-MiniLM-L6-v2` soporta una ventana máxima de 256 *tokens*, la inmensa mayoría de los documentos se procesarán en su totalidad. 

Cabe destacar que, al construir el documento, la sinopsis se ha colocado deliberadamente al final del texto. De este modo, incluso si ocurre un truncamiento, el modelo ya habrá procesado todos los metadatos clave y el planteamiento inicial de la trama, información que esperablemente es suficiente para capturar la esencia de la obra.

### Generar embeddings

A continuación, procederemos a la generación de los *embeddings* para cada ítem. Tal y como se recomienda en la práctica, utilizaremos el modelo ligero `all-MiniLM-L6-v2` de la biblioteca `sentence-transformers`. Este modelo procesará la lista de documentos y transformará cada uno en un vector denso de 384 dimensiones.

In [12]:
model = SentenceTransformer('all-MiniLM-L6-v2')
documents = df['document'].tolist()
embeddings = model.encode(documents, show_progress_bar=True, batch_size=64)
embeddings = np.array(embeddings).astype('float32')
print(f'Forma de los embeddings: {embeddings.shape}')  

Batches: 100%|██████████| 319/319 [03:39<00:00,  1.45it/s]

Forma de los embeddings: (20370, 384)


Una vez generados los *embeddings*, procederemos a construir el índice de búsqueda vectorial utilizando la biblioteca FAISS. Como queremos medir qué tan similares son los *embeddings* entre sí usando la similitud del coseno, primero normalizamos los vectores. Gracias a esto, podemos usar un índice de producto escalar (`IndexFlatIP`) ya que al normalizar los vectores el producto escalar es equivlente a la similitud coseno.

### Normalizar e indexar

In [13]:
# Normalizar a norma 1 para que el producto escalar == similitud coseno
faiss.normalize_L2(embeddings)

d = embeddings.shape[1]  # 384
index = faiss.IndexFlatIP(d)
index.add(embeddings)

print(f"Vectores indexados: {index.ntotal}")

Vectores indexados: 20370


### Guardar

In [14]:
# Guardar el índice FAISS
faiss.write_index(index, 'data/anime_index.faiss')

# Guardar el dataframe con los documentos y metadatos
df.to_csv('data/anime_processed.csv', index=False)

# Guardar los embeddings
np.save('data/anime_embeddings.npy', embeddings)

## Sesión 2

In [ ]:
import numpy as np
import pandas as pd
import faiss

### Cargar los datos

En primer lugar, cargamos el dataframe con documentos y metadatos generado en la sesión anterior, así como los embeddings y el índice FAISS. También cargaremos el archivo `users-score-2023.csv`, que contiene los ratings dados por los usuarios para generar los perfiles de usuarios.

In [16]:
df = pd.read_csv('data/anime_processed.csv')
index = faiss.read_index('data/anime_index.faiss')
embeddings = np.load('data/anime_embeddings.npy')

ratings = pd.read_csv('data/users-score-2023.csv')

print("Dimensiones de dataframe de ratings:", ratings.shape)
print(ratings['rating'].value_counts().sort_index())
print(f"Usuarios únicos: {ratings['user_id'].nunique()}")
print(f"Animes puntuados únicos: {ratings['anime_id'].nunique()}")

Dimensiones de dataframe de ratings: (24325191, 5)
rating
1       98069
2      132092
3      233675
4      562822
5     1379480
6     2766482
7     5452152
8     6060484
9     4429914
10    3210021
Name: count, dtype: int64
Usuarios únicos: 270033
Animes puntuados únicos: 16500


### Creación de los perfiles de usuario

Para garantizar la calidad de las recomendaciones y poder construir perfiles sólidos, haremos un filtrado previo de las interacciones. En este paso, eliminaremos a los usuarios con muy poca actividad, conservando únicamente a aquellos que hayan valorado al menos 12 animes.

In [17]:
# Contar cuántos animes ha valorado cada usuario
user_counts = ratings.groupby('user_id').size()

# Quedarse con usuarios que hayan valorado al menos 12 animes
valid_users = user_counts[user_counts >= 12].index

ratings_filtered = ratings[ratings['user_id'].isin(valid_users)] # ratings de los usuarios válidos
print(f"Usuarios válidos: {len(valid_users)}")

Usuarios válidos: 186245


Para evaluar el rendimiento de nuestro sistema de recomendación, seleccionaremos perfiles de usuario con gustos muy definidos. Cruzaremos las valoraciones altamente positivas (puntuación de 8 o superior) con la información de los géneros. El objetivo es identificar a un usuario representativo para varias categorías principales (como Acción, Romance o Fantasía), eligiendo a aquel que acumule la mayor proporción de notas altas (con respecto al total de reseñas positivas) en dicho género.

In [18]:
# Seleccionar usuarios candidatos por género: para cada género objetivo, encontrar usuarios que hayan valorado animes de ese género con nota >= 8
anime_genres = df[['anime_id', 'Genres']].copy()
ratings_genres = ratings_filtered.merge(anime_genres, on='anime_id', how='inner') # Merge ligero solo con géneros
ratings_genres = ratings_genres[ratings_genres['rating'] >= 8] # filtrar valoraciones positivas

print(f"Shape tras filtrar >= 8: {ratings_genres.shape}")

target_genres = ['Adventure', 'Action', 'Romance', 'Fantasy', 'Sci-Fi', 'Sports', 'Suspense']
selected_users = []
used_ids = set()

# Calculamos el total de valoraciones positivas (>= 8) que tiene cada usuario
total_positive_per_user = ratings_genres.groupby('user_id').size()

for genre in target_genres:
    # Cuántos animes de este género ha valorado bien cada usuario
    genre_ratings = ratings_genres[ratings_genres['Genres'].str.contains(genre, case=False, na=False)]
    genre_counts = genre_ratings.groupby('user_id').size() # número de animes del género valorados positivamente por cada usuario
    
    # Calcular la proporción (Animes positivos del género / Total de animes positivos)
    proportions = (genre_counts / total_positive_per_user).dropna()
    
    # Excluir usuarios ya seleccionados para que no se repitan
    proportions = proportions[~proportions.index.isin(used_ids)]
    
    # Ordenar para quedarnos con el usuario que tenga el mayor porcentaje
    proportions = proportions.sort_values(ascending=False)
    
    if len(proportions) > 0:
        top_user = proportions.index[0]
        selected_users.append({
            'user_id': top_user, 
            'top_genre': genre,
            'genre_ratio': f"{proportions.iloc[0] * 100:.1f}%"
        })
        used_ids.add(top_user)

selected_users = pd.DataFrame(selected_users)
print(selected_users)

Shape tras filtrar >= 8: (13414635, 6)
   user_id  top_genre genre_ratio
0   360537  Adventure      100.0%
1   104074     Action      100.0%
2  1286221    Romance      100.0%
3   338673    Fantasy      100.0%
4   537333     Sci-Fi      100.0%
5  1099623     Sports      100.0%
6  1226789   Suspense      100.0%


In [19]:
# Mostrar animes valorados positivamente por cada usuario seleccionado
for _, row in selected_users.iterrows():
    user_id = row['user_id']
    top_genre = row['top_genre']
    
    print(f"\nUsuario {user_id}. Género favorito: {top_genre}")
    
    # Animes valorados positivamente por este usuario
    user_ratings = ratings_genres[(ratings_genres['user_id'] == user_id) & (ratings_genres['Genres'].str.contains(top_genre, case=False, na=False))]
    
    # Mostrar los títulos de esos animes
    anime_ids = user_ratings['anime_id'].unique()
    anime_titles = df[df['anime_id'].isin(anime_ids)]['English name'].tolist()
    
    print(f"Animes valorados positivamente en {top_genre}:")
    for title in anime_titles:
        print(f"- {title}")


Usuario 360537. Género favorito: Adventure
Animes valorados positivamente en Adventure:
- naruto
- princess mononoke
- spirited away
- bleach
- howl's moving castle
- naruto the movie 1: ninja clash in the land of snow
- kiki's delivery service
- my neighbor totoro
- naruto: the lost story - mission: protect the waterfall village
- naruto the movie 2: legend of the stone of gelel
- naruto shippuden
- naruto the movie 3: guardians of the crescent moon kingdom

Usuario 104074. Género favorito: Action
Animes valorados positivamente en Action:
- naruto
- fullmetal alchemist
- hunter x hunter
- hunter x hunter: greed island
- hunter x hunter: greed island final
- shaman king
- elfen lied
- bleach

Usuario 1286221. Género favorito: Romance
Animes valorados positivamente en Romance:
- please teacher!
- tokimeki memorial
- my bride is a mermaid
- rosario + vampire capu2
- haganai: i don't have many friends
- sword art online

Usuario 338673. Género favorito: Fantasy
Animes valorados positivam

#### Estrategia de representación del perfil de usuario

El guión sugiere construir el perfil concatenando los documentos de los ítems consumidos por cada usuario. Sin embargo, esta estrategia presenta un problema importante: el modelo `all-MiniLM-L6-v2` tiene una ventana de contexto de 256 *tokens*, y la concatenación de las sinopsis y metadatos de 10-15 animes supera con creces ese límite. El resultado sería que el modelo solo procesaría el texto del primer o segundo anime del perfil, ignorando el resto y sesgando la representación.

Para resolverlo, se ha optado por representar cada perfil como la **media ponderada por *rating* de los *embeddings* individuales** de los ítems que el usuario ha consumido. Esta estrategia presenta las siguientes ventajas:
- Cada anime contribuye al perfil con su representación completa, sin pérdida de información.
- Los animes con mayor valoración tienen más peso.
- Es computacionalmente eficiente, ya que reutiliza los *embeddings* ya calculados en la sesión anterior.
- Es coherente con la esencia del guión, que describe el perfil como una agregación del contenido consumido.

In [20]:
target_user_ids = selected_users['user_id'].tolist() # lista de los usuarios seleccionados

# Filtrar las valoraciones para quedarnos solo con las de los usuarios seleccionados y con nota >= 8
ratings_target = ratings_filtered[
    (ratings_filtered['user_id'].isin(target_user_ids)) &
    (ratings_filtered['rating'] >= 8)
]

# Limitar a los top 15 animes mejor valorados por usuario
ratings_target = (
    ratings_target
    .sort_values(['user_id', 'rating'], ascending=[True, False])
    .groupby('user_id')
    .head(15)
)

# Merge con documentos solo para estos usuarios
ratings_with_docs = ratings_target.merge(
    df[['anime_id', 'document']],
    on='anime_id',
    how='inner'
)

In [21]:
# Mapea de anime_id a posición en el array de embeddings
anime_id_to_idx = {aid: i for i, aid in enumerate(df['anime_id'].values)}

def build_profile_vector(user_id, ratings_with_docs, embeddings, anime_id_to_idx):
    """
    Construye el vector de perfil de un usuario como la media ponderada por rating de los embeddings de los ítems que ha consumido.
    """
    user_data = ratings_with_docs[ratings_with_docs['user_id'] == user_id]
    
    vecs, weights = [], []
    for _, row in user_data.iterrows():
        idx = anime_id_to_idx.get(row['anime_id']) # obtener el índice del anime en el array de embeddings
        if idx is not None:
            vecs.append(embeddings[idx]) # añadir el embedding del anime a la lista de vectores
            weights.append(float(row['rating'])) # añadir el rating como peso
    
    if not vecs:
        return None
    
    vecs = np.array(vecs, dtype=np.float32) # convertir a array de numpy
    weights = np.array(weights, dtype=np.float32) # convertir a array de numpy
    weights /= weights.sum()  # normalizar pesos para que sumen 1
    
    # Calcular embedding de perfil como media ponderada de los embeddings de los animes valorados positivamente
    profile_vec = np.average(vecs, weights=weights, axis=0).astype(np.float32)
    return profile_vec


# Construir la matriz de perfiles
profile_vectors = []
valid_user_ids  = []

# Para cada usuario seleccionado, construir su vector de perfil
for uid in target_user_ids:
    vec = build_profile_vector(uid, ratings_with_docs, embeddings, anime_id_to_idx)
    if vec is not None:
        profile_vectors.append(vec)
        valid_user_ids.append(uid)

profile_matrix = np.array(profile_vectors)  # shape: (n_users, 384)
print(f"Perfiles construidos: {profile_matrix.shape}")

Perfiles construidos: (7, 384)


### Recuperación con FAISS

Con los perfiles de usuario ya construidos como vectores densos, procedemos a la fase de recuperación. Normalizamos la matriz de perfiles (necesario para que el producto escalar con `IndexFlatIP` sea equivalente a la similitud coseno) y lanzamos una búsqueda para obtener los *top-k* ítems más similares a cada perfil.

Recuperamos los **top-20** candidatos (excluyendo los que ya se encuentran en el perfil), de los cuales el LLM seleccionará y reordenará los 3 mejores en la sesión siguiente. Recuperar más candidatos de los que finalmente se recomiendan proporciona al LLM un contexto más amplio para elegir con mayor criterio.

In [22]:
# Normalizar perfiles para similitud coseno
faiss.normalize_L2(profile_matrix)

K_SEARCH = 30  # candidatos a recuperar por usuario
K = 20  # candidatos que realmente se van a utilizar
scores, indices = index.search(profile_matrix, K_SEARCH)
# scores:  (n_users, K): similitud coseno con cada candidato
# indices: (n_users, K): posición en el dataframe de cada candidato

print(f"Búsqueda completada. Forma de la matriz de resultados: {indices.shape}")

Búsqueda completada. Forma de la matriz de resultados: (7, 30)


In [23]:
# IDs de animes en el perfil de cada usuario
profile_ids_per_user = (
    ratings_with_docs
    .groupby('user_id')['anime_id']
    .apply(set)
    .to_dict()
)

# Filtrar los ítems del perfil de los resultados recuperados
filtered_indices = []
filtered_scores  = []

for i, uid in enumerate(valid_user_ids):
    profile_ids = profile_ids_per_user.get(uid, set())
    
    clean_idx, clean_scores = [], []
    for idx, score in zip(indices[i], scores[i]):
        anime_id = df.iloc[idx]['anime_id']
        if anime_id not in profile_ids:
            clean_idx.append(idx)
            clean_scores.append(score)
        if len(clean_idx) == K:  # parar cuando tenemos K candidatos limpios
            break
    if len(clean_idx) != K:
        print(f"Advertencia: Usuario {uid} tiene solo {len(clean_idx)} candidatos limpios. Aumenta K_SEARCH.")
        
    filtered_indices.append(clean_idx)
    filtered_scores.append(clean_scores)

indices = np.array(filtered_indices)
scores = np.array(filtered_scores)

### Visualización de los resultados de recuperación

In [24]:
user_genre_map = dict(zip(selected_users['user_id'], selected_users['top_genre']))

for i, uid in enumerate(valid_user_ids):
    genre = user_genre_map.get(uid, '?')
    print(f"\n{'-'*60}")
    print(f"Usuario {uid}  |  Género preferido: {genre}")
    print(f"{'-'*60}")

    retrieved_rows = df.iloc[indices[i]][['English name', 'Genres', 'Era']].copy()
    retrieved_rows['similarity'] = scores[i].round(4)
    retrieved_rows.index = range(1, K + 1)
    print(retrieved_rows.to_string())


------------------------------------------------------------
Usuario 360537  |  Género preferido: Adventure
------------------------------------------------------------
                                                                   English name                                             Genres            Era  similarity
1                                                                        naruto                 action, adventure, comedy, fantasy    2020s anime      0.7901
2                                           naruto shippuden the movie 2: bonds                         action, adventure, fantasy    2000s anime      0.7628
3                                        naruto shippuden the movie 7: the last                action, adventure, fantasy, romance    2010s anime      0.7576
4                                                                    haguregumo                                             comedy  classic anime      0.7415
5                                       

### Guardar resultados de recuperación

In [25]:
np.save('data/retrieval_scores.npy', scores)
np.save('data/retrieval_indices.npy', indices)
np.save('data/valid_user_ids.npy', np.array(valid_user_ids))
selected_users.to_csv('data/selected_users.csv', index=False)

ratings_with_docs.to_csv('data/ratings_with_docs.csv', index=False)

## Sesión 3

Aumentación, generación con LLM, filtro de alucinaciones y evaluación.

In [ ]:
import pandas as pd
import numpy as np
import json
import re
import google.generativeai as genai
from dotenv import load_dotenv
import os

C:\Users\Laura\AppData\Local\Temp\ipykernel_24120\1115421336.py:5: FutureWarning: 

All support for the `google.generativeai` package has ended. It will no longer be receiving 
updates or bug fixes. Please switch to the `google.genai` package as soon as possible.
See README for more details:

https://github.com/google-gemini/deprecated-generative-ai-python/blob/main/README.md

  import google.generativeai as genai


In [ ]:
load_dotenv()

### Cargar datos y resultados de recuperación

In [27]:
# Dataframe de animes procesado en la sección 1, con columna de documento
df = pd.read_csv('data/anime_processed.csv')

# Resultados de la búsqueda de los arrays de usuario:
# - scores:  (n_users, K): similitud coseno con cada candidato
# - indices: (n_users, K): posición en el dataframe df de cada candidato
scores  = np.load('data/retrieval_scores.npy')
indices = np.load('data/retrieval_indices.npy')

# Ids de los usuarios seleccionados
valid_user_ids = np.load('data/valid_user_ids.npy').tolist()

# Dataframe de los usuarios seleccionados con las columnas: user_id, top_genre y genre_ratio
selected_users = pd.read_csv('data/selected_users.csv') 

# Ratings de los 15 mejor animes valorados por cada usuario seleccionado, con columna de documento
ratings_with_docs  = pd.read_csv('data/ratings_with_docs.csv') 

print('Datos cargados correctamente.')
print(f'Usuarios: {len(valid_user_ids)}')

Datos cargados correctamente.
Usuarios: 7


In [ ]:
# Ratings de los 5 mejor animes valorados por cada usuario seleccionado
ratings_top5 = (
    ratings_with_docs
    .sort_values(['user_id', 'rating'], ascending=[True, False])
    .groupby('user_id')
    .head(5)
)

# Dataframe con los 5 documentos mejor valorado para cada usuario seleccionado
user_profile_text = (
    ratings_top5
    .groupby('user_id')
    .apply(lambda g: ' | '.join(g['document'].tolist()))
    .reset_index()
    .rename(columns={0: 'profile_text'})
)

### Aumentación y Generación

Para la fase de generación se utiliza la **API de Google Gemini** con el modelo `gemini-2.5-flash-lite`. Este modelo es accesible sin descargar ningún peso localmente y cuenta con una ventana de contexto amplia (1 millón de *tokens*), más que suficiente para prompts con múltiples candidatos.

Solo es necesaria una **API key gratuita de Google AI Studio**, obtenida en en https://aistudio.google.com/apikey.

Para cada usuario se construye un prompt que incluye:
1. El **rol** del sistema (sistema de recomendación de anime).
2. El **perfil del usuario** (sus 5 animes mejor valorados).
3. La **lista de candidatos** recuperados por FAISS.
4. Las **instrucciones** de re-ranking, justificación, filtro de calidad e integridad.
5. El **formato de salida** esperado (JSON estricto).


In [ ]:
genai.configure(api_key=genai.configure(api_key=os.getenv("GEMINI_API_KEY")))

LLM_MODEL = 'gemini-2.5-flash-lite'
TOP_K_LLM = 10

client_gemini = genai.GenerativeModel(
    model_name=LLM_MODEL,
    system_instruction=(
        'Actúas como un sistema de recomendación de anime. '
        'Respondes siempre en JSON válido y considerando únicamente los ítems candidatos que se te dan.'
    ),
    generation_config=genai.GenerationConfig(
        temperature=0.3,
        max_output_tokens=800,
    ),
)

In [80]:
def build_prompt(profile_text, candidate_docs):
    """
    Construye el prompt aumentado para el LLM.
    """
    candidates_str = '\n\n'.join(
        f"{j}. {doc}" for j, doc in enumerate(candidate_docs, start=1)
    )

    prompt = f"""Actúas como un sistema de recomendación de anime. Tu objetivo es analizar el perfil \
de un usuario y seleccionar, de entre una lista de candidatos, las opciones que mejor encajen con sus gustos actuales.\

El usuario ha definido sus preferencias actuales como: {profile_text}

A continuación se presentan los ítems más similares encontrados en nuestro catálogo: {candidates_str}

INSTRUCCIONES:
1. RE-RANKING: Ordena los 3 mejores ítems de mayor a menor afinidad real con el perfil del usuario.
2. JUSTIFICACIÓN: Para cada ítem, explica en una frase corta el motivo por el que encaja, mencionando detalles específicos de la sinopsis o demás metadatos.
3. FILTRO DE CALIDAD: Si un ítem recuperado es totalmente irrelevante para el perfil, ignóralo.
4. INTEGRIDAD. No inventes animes que no estén en la lista de candidatos.

Devuelve la respuesta estrictamente en este formato JSON, sin ningún texto adicional antes o después:
[
  {{"ranking": 1, "titulo": "Nombre del anime", "razonamiento": "Explicación breve..."}},
  {{"ranking": 2, "titulo": "Nombre del anime", "razonamiento": "Explicación breve..."}},
  {{"ranking": 3, "titulo": "Nombre del anime", "razonamiento": "Explicación breve..."}}
]"""
    return prompt

In [81]:
def generate_recommendations(profile_text, candidate_docs):
    prompt = build_prompt(profile_text, candidate_docs)
    response = client_gemini.generate_content(prompt)
    return response.text.strip()


def parse_llm_output(raw_text):
    """Extrae y parsea el JSON de la respuesta del LLM."""
    # Eliminar bloques de código markdown si el LLM los añade
    clean = re.sub(r'```(?:json)?', '', raw_text).strip().rstrip('`').strip()

    # Buscar el array JSON en el texto (por si hay texto adicional)
    match = re.search(r'\[.*\]', clean, re.DOTALL)
    if match:
        clean = match.group(0)

    return json.loads(clean)

In [ ]:
all_results = []

for i, uid in enumerate(valid_user_ids):
    print(f"\n{'-'*60}")
    print(f"Procesando usuario {uid}...")
    
    # Recuperar candidatos (top-K_LLM del ranking FAISS)
    top_indices = indices[i][:TOP_K_LLM]
    top_scores = scores[i][:TOP_K_LLM]
    
    # Extraer los documentos y títulos de esos candidatos
    candidate_docs = df.iloc[top_indices]['document'].tolist()
    faiss_titles = df.iloc[top_indices]['English name'].tolist()
    print(f"  Candidatos recuperados (FAISS): {faiss_titles}")
    
    # Perfil textual del usuario
    profile_row = user_profile_text[user_profile_text['user_id'] == uid]
    profile_text = profile_row['profile_text'].values[0] if len(profile_row) > 0 else "Sin perfil disponible" # extraer el texto del perfil para este usuario

    # Llamada al LLM
    raw_output = None
    try:
        raw_output = generate_recommendations(profile_text, candidate_docs)
        llm_results = parse_llm_output(raw_output)
        parse_ok = True
    except Exception as e:
        print(f"\tError: Fallo en LLM o parsing: {e}")
        if raw_output:
            print(f"\tRespuesta bruta: {raw_output[:500]}")
        llm_results = []
        parse_ok = False
    
    all_results.append({
        'user_id': uid,
        'faiss_titles': faiss_titles,
        'faiss_scores': list(top_scores),
        'llm_results': llm_results,
        'parse_ok': parse_ok
    })
    
    # --- Mostrar resultado ---
    if parse_ok:
        # Títulos del perfil
        profile_animes = (
            ratings_top5[ratings_top5['user_id'] == uid]
            .merge(df[['anime_id', 'English name']], on='anime_id', how='left')['English name']
            .tolist()
        )
        top_genre = selected_users[selected_users['user_id'] == uid]['top_genre'].values[0]
        
        print(f"  Género preferido: {top_genre}")
        print(f"  Perfil (top 5 animes):")
        for title in profile_animes:
            print(f"    - {title}")
        print(f"  Recomendaciones del LLM:")
        for rec in llm_results:
            print(f"    {rec['ranking']}. {rec['titulo']}")
            print(f"       Justificación: {rec['razonamiento']}")


------------------------------------------------------------
Procesando usuario 360537...
  Candidatos recuperados (FAISS): ['naruto', 'naruto shippuden the movie 2: bonds', 'naruto shippuden the movie 7: the last', 'haguregumo', 'boruto: naruto the movie', 'naruto shippuden the movie 6: road to ninja', 'otogi zoshi: the legend of magatama', 'crayon shin-chan movie 10: arashi wo yobu appare! sengoku daikassen', 'ninja girl & samurai master', 'boruto: naruto next generations']
  Género preferido: Adventure
  Perfil (top 5 animes):
    - bleach
    - naruto shippuden
    - howl's moving castle
    - kiki's delivery service
    - princess mononoke
  Recomendaciones del LLM:
    1. naruto shippuden the movie 2: bonds
       Justificación: Comparte géneros, estudio, era y rating con Bleach y Naruto Shippuden, además de ser una película del universo Naruto.
    2. boruto: naruto the movie
       Justificación: Similar a Naruto Shippuden en géneros, estudio, era y rating, y se enfoca en la c

### Filtro de alucinaciones

Comprobamos que todos los títulos devueltos por el LLM estaban efectivamente en la lista de candidatos de entrada. Para ello realizamos una comparación insensible a mayúsculas/minúsculas. Si el LLM devuelve un título que no figura en la lista, se registra como alucinación.

In [ ]:
def check_hallucinations(faiss_titles, llm_results):
    faiss_lower = {t.lower().strip() for t in faiss_titles}
    valid, hallucinated = [], []
    
    for rec in llm_results:
        title_lower = rec['titulo'].lower().strip()
        if title_lower in faiss_lower:
            valid.append(rec)
        else:
            hallucinated.append(rec)
    
    return valid, hallucinated


print("\n" + "-"*60)
print("INFORME DE ALUCINACIONES")
print("-"*60)

total_recs = 0
total_hallucinations = 0

for result in all_results:
    if not result['parse_ok'] or not result['llm_results']:
        continue
    
    uid = result['user_id']
    valid, hallucinated = check_hallucinations(result['faiss_titles'], result['llm_results'])
    
    total_recs += len(result['llm_results'])
    total_hallucinations += len(hallucinated)
    
    # Actualizar con solo recomendaciones válidas
    result['llm_results_clean'] = valid
    result['hallucinated'] = hallucinated
    
    status = "Sin alucinaciones" if not hallucinated else f"{len(hallucinated)} alucinación(es):"
    print(f"\nUsuario {uid}: {status}")
    if hallucinated:
        for h in hallucinated:
            print(f"- Título inventado: '{h['titulo']}'")

print(f"\nResumen: {total_hallucinations}/{total_recs} recomendaciones fueron alucinaciones ({100*total_hallucinations/max(total_recs,1):.1f}%)")


------------------------------------------------------------
INFORME DE ALUCINACIONES
------------------------------------------------------------

Usuario 360537: Sin alucinaciones

Usuario 104074: Sin alucinaciones

Usuario 1286221: 1 alucinación(es):
- Título inventado: 'please teacher!'

Usuario 338673: Sin alucinaciones

Usuario 537333: Sin alucinaciones

Usuario 1099623: Sin alucinaciones

Usuario 1226789: Sin alucinaciones

Resumen: 1/21 recomendaciones fueron alucinaciones (4.8%)


### Análisis del re-ranking

Comparamos el orden propuesto por FAISS con el orden final elegido por el LLM. Para cada usuario, calculamos cuántas posiciones ha movido el LLM cada ítem con respecto al ranking original de recuperación.

In [107]:
print("-"*60)
print("ANÁLISIS DE RE-RANKING")
print("-"*60)

reranking_changes = []  # desplazamientos de posición para cada ítem

for result in all_results:
    if not result['parse_ok']:
        continue
    
    uid = result['user_id']
    faiss_titles = result['faiss_titles']   # posición 0 = más similar según FAISS
    llm_results = result.get('llm_results_clean', result['llm_results'])
    
    # Mapeo de título -> posición FAISS (1-indexed)
    faiss_pos = {t.lower().strip(): rank+1 for rank, t in enumerate(faiss_titles)}
    
    print(f"\nUsuario {uid}:")
    print(f"  {'Rank LLM':<10} {'Rank FAISS':<12} {'Cambio':<10} Título")
    print(f"  {'-'*55}")
    
    for rec in llm_results:
        title_lower = rec['titulo'].lower().strip()
        faiss_rank = faiss_pos.get(title_lower, None)
        llm_rank = rec['ranking']
        
        if faiss_rank is not None:
            delta = faiss_rank - llm_rank   # positivo indique que el LLM subió el ítem
            reranking_changes.append(abs(delta))
            change_str = f"+{delta}" if delta >= 0 else str(delta)
        else:
            change_str = "N/A"
            faiss_rank = "?"
        
        print(f"  {llm_rank:<10} {str(faiss_rank):<12} {change_str:<10} {rec['titulo']}")

if reranking_changes:
    print(f"\nDesplazamiento medio de posición: {np.mean(reranking_changes):.2f} puestos")
    print(f"Máximo desplazamiento observado: {max(reranking_changes)} puestos")
    changed = sum(1 for d in reranking_changes if d > 0)
    print(f"Ítems cuyo orden cambió respecto a FAISS: {changed}/{len(reranking_changes)} ({100*changed/len(reranking_changes):.0f}%)")

------------------------------------------------------------
ANÁLISIS DE RE-RANKING
------------------------------------------------------------

Usuario 360537:
  Rank LLM   Rank FAISS   Cambio     Título
  -------------------------------------------------------
  1          2            +1         naruto shippuden the movie 2: bonds
  2          5            +3         boruto: naruto the movie
  3          6            +3         naruto shippuden the movie 6: road to ninja

Usuario 104074:
  Rank LLM   Rank FAISS   Cambio     Título
  -------------------------------------------------------
  1          1            +0         hunter x hunter
  2          5            +3         shakugan no shana: season i
  3          7            +4         dororo

Usuario 1286221:
  Rank LLM   Rank FAISS   Cambio     Título
  -------------------------------------------------------
  1          6            +5         rosario + vampire
  3          1            -2         kokoro connect

Usuario 338

### Recomendaciones finales (post filtro de alucinaciones)

In [101]:
print("\nRECOMENDACIONES FINALES")

for result in all_results:
    uid = result['user_id']

    llm_clean = result.get('llm_results_clean', result.get('llm_results', []))

    # Recuperar género dominante
    genre_row = selected_users[selected_users['user_id'] == uid]
    genre = genre_row['top_genre'].values[0] if len(genre_row) > 0 else '?'

    # Recuperar top 5 animes preferidos del usuario
    profile_animes = (
        ratings_top5[ratings_top5['user_id'] == uid]
        .merge(
            df[['anime_id', 'English name']],
            on='anime_id',
            how='left'
        )['English name']
        .tolist()
    )

    print(f"{'-' * 60}")
    print(f"Usuario {uid}")
    print(f"  Género preferido: {genre}")

    print("  Perfil (top 5 animes):")
    if profile_animes:
        for title in profile_animes:
            print(f"    - {title}")
    else:
        print("    No disponible.")

    print("\n  Recomendaciones del LLM:")

    if not llm_clean:
        print("    No se obtuvieron recomendaciones válidas.")
        continue

    for rec in llm_clean:
        print(f"    {rec['ranking']}. {rec['titulo']}")
        print(f"       Justificación: {rec['razonamiento']}")
    print("\n")


RECOMENDACIONES FINALES
------------------------------------------------------------
Usuario 360537
  Género preferido: Adventure
  Perfil (top 5 animes):
    - bleach
    - naruto shippuden
    - howl's moving castle
    - kiki's delivery service
    - princess mononoke

  Recomendaciones del LLM:
    1. naruto shippuden the movie 2: bonds
       Justificación: Comparte géneros, estudio, era y rating con Bleach y Naruto Shippuden, además de ser una película del universo Naruto.
    2. boruto: naruto the movie
       Justificación: Similar a Naruto Shippuden en géneros, estudio, era y rating, y se enfoca en la continuación de la saga Naruto.
    3. naruto shippuden the movie 6: road to ninja
       Justificación: Comparte géneros, estudio, era y rating con Bleach y Naruto Shippuden, y es una película dentro del universo Naruto.


------------------------------------------------------------
Usuario 104074
  Género preferido: Action
  Perfil (top 5 animes):
    - elfen lied
    - hunter